# 02 - Líder por Rodada → Campeão?

A partir de qual rodada o líder do Brasileirão é praticamente campeão?
Calculamos a classificação rodada a rodada e verificamos se o líder de cada rodada acabou campeão.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)
df = df[df["ano"] >= 2003].copy()

In [2]:
def calcular_classificacao(df_season):
    """Calcula a classificação acumulada rodada a rodada para uma temporada."""
    teams = set(df_season["mandante"].unique()) | set(df_season["visitante"].unique())
    
    # Inicializar pontos, vitorias, saldo de gols
    pontos = {t: 0 for t in teams}
    vitorias = {t: 0 for t in teams}
    saldo = {t: 0 for t in teams}
    gols_pro = {t: 0 for t in teams}
    
    max_rodada = df_season["rodata"].max()
    lideres = {}  # rodada -> time líder
    
    for rodada in range(1, max_rodada + 1):
        jogos = df_season[df_season["rodata"] == rodada]
        
        for _, jogo in jogos.iterrows():
            m, v = jogo["mandante"], jogo["visitante"]
            gm, gv = jogo["mandante_Placar"], jogo["visitante_Placar"]
            
            if pd.isna(gm) or pd.isna(gv):
                continue
            
            gm, gv = int(gm), int(gv)
            saldo[m] += gm - gv
            saldo[v] += gv - gm
            gols_pro[m] += gm
            gols_pro[v] += gv
            
            if gm > gv:
                pontos[m] += 3
                vitorias[m] += 1
            elif gm < gv:
                pontos[v] += 3
                vitorias[v] += 1
            else:
                pontos[m] += 1
                pontos[v] += 1
        
        # Classificação: pontos > vitórias > saldo > gols pró
        ranking = sorted(teams, key=lambda t: (pontos[t], vitorias[t], saldo[t], gols_pro[t]), reverse=True)
        lideres[rodada] = ranking[0]
    
    campeao = lideres.get(max_rodada, None)
    return lideres, campeao

In [3]:
# Calcular para cada temporada
resultados = []

for ano in sorted(df["ano"].unique()):
    df_season = df[df["ano"] == ano]
    max_rodada = df_season["rodata"].max()
    
    # Pular temporadas incompletas ou com formato diferente
    if max_rodada < 30:
        continue
    
    lideres, campeao = calcular_classificacao(df_season)
    
    for rodada, lider in lideres.items():
        resultados.append({
            "ano": ano,
            "rodada": rodada,
            "lider": lider,
            "campeao": campeao,
            "lider_foi_campeao": lider == campeao
        })

df_resultados = pd.DataFrame(resultados)
print(f"Temporadas analisadas: {df_resultados['ano'].nunique()}")
print(f"\nCampeões:")
for ano in sorted(df_resultados["ano"].unique()):
    c = df_resultados[df_resultados["ano"] == ano]["campeao"].iloc[0]
    print(f"  {ano}: {c}")

Temporadas analisadas: 22

Campeões:
  2003: Cruzeiro
  2004: Santos
  2005: Corinthians
  2006: Sao Paulo
  2007: Sao Paulo
  2008: Sao Paulo
  2009: Flamengo
  2010: Fluminense
  2011: Corinthians
  2012: Fluminense
  2013: Cruzeiro
  2014: Cruzeiro
  2015: Corinthians
  2016: Palmeiras
  2017: Corinthians
  2018: Palmeiras
  2019: Flamengo
  2021: Atletico-MG
  2022: Palmeiras
  2023: Palmeiras
  2024: Botafogo-RJ
  2025: Flamengo


In [4]:
# Probabilidade do líder na rodada X ser campeão
prob_por_rodada = df_resultados.groupby("rodada")["lider_foi_campeao"].mean() * 100
prob_df = prob_por_rodada.reset_index()
prob_df.columns = ["rodada", "probabilidade"]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=prob_df["rodada"],
    y=prob_df["probabilidade"],
    mode="lines+markers",
    name="Probabilidade",
    line=dict(color="#1a7a1a", width=3),
    marker=dict(size=6),
    hovertemplate="Rodada %{x}<br>Probabilidade: %{y:.1f}%<extra></extra>"
))

# Linha de referência em 50%
fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=80, line_dash="dash", line_color="red", opacity=0.3,
              annotation_text="80%", annotation_position="right")

fig.update_layout(
    title="Probabilidade do Líder na Rodada X Ser Campeão<br><sup>Brasileirão Série A (2003-presente)</sup>",
    xaxis_title="Rodada",
    yaxis_title="Probabilidade (%)",
    yaxis=dict(range=[0, 105]),
    template="plotly_white",
    width=900,
    height=500
)

fig.show()
fig.write_html("../charts/lider_por_rodada.html", include_plotlyjs="cdn")
print("Gráfico exportado: charts/lider_por_rodada.html")

Gráfico exportado: charts/lider_por_rodada.html


In [5]:
# Detalhamento: quem era líder em cada rodada de cada temporada
pivot = df_resultados.pivot(index="ano", columns="rodada", values="lider")

# Heatmap: líder foi campeão (1) ou não (0)
pivot_bool = df_resultados.pivot(index="ano", columns="rodada", values="lider_foi_campeao")
# Preencher NaN com False antes de converter para int (temporadas com rodadas diferentes)
pivot_bool = pivot_bool.fillna(False).astype(int)

fig2 = px.imshow(
    pivot_bool,
    labels=dict(x="Rodada", y="Ano", color="Líder = Campeão"),
    color_continuous_scale=["#ff4444", "#22aa22"],
    title="Líder na Rodada Foi o Campeão?<br><sup>Verde = Sim, Vermelho = Não</sup>",
    aspect="auto"
)
fig2.update_layout(width=1000, height=600, template="plotly_white")

# Adicionar texto no hover com nome do líder
customdata = pivot.values
fig2.update_traces(
    hovertemplate="Ano: %{y}<br>Rodada: %{x}<br>Líder: %{customdata}<extra></extra>",
    customdata=customdata
)

fig2.show()
fig2.write_html("../charts/lider_heatmap.html", include_plotlyjs="cdn")
print("Gráfico exportado: charts/lider_heatmap.html")

Gráfico exportado: charts/lider_heatmap.html


In [6]:
# Insight: a partir de qual rodada a probabilidade passa de 80%?
rodada_80 = prob_df[prob_df["probabilidade"] >= 80]["rodada"].min()
rodada_90 = prob_df[prob_df["probabilidade"] >= 90]["rodada"].min()

print(f"A partir da rodada {rodada_80}, o líder tem >= 80% de chance de ser campeão")
if pd.notna(rodada_90):
    print(f"A partir da rodada {rodada_90}, o líder tem >= 90% de chance de ser campeão")

A partir da rodada 22, o líder tem >= 80% de chance de ser campeão
A partir da rodada 36, o líder tem >= 90% de chance de ser campeão
